# 04 跟踪后处理

本 Notebook 对应 **scripts/tracking**：SLEAP 导出 CSV 后的 **跨帧 2D 个体追踪**、异常值处理、缺失值填充与可视化，为后续 3D 重建提供每视角内一致的 a/b/c 轨迹。

## 流程概览（跨帧 3D 个体追踪的 2D 部分）

1. **自动生成 file_sets.json**：扫描并分组同一实验的 3 个视角文件（PLAYER1/2/3）
2. **ID 分配**：Hungarian 算法 + 可选运动方向代价，为每条鱼分配唯一且一致的 ID（视频内跨帧不交换）
3. **异常值处理**：按帧间隔缩放阈值（1 帧内跳动 > 100 像素、2 帧 > 200 像素，以此类推），去除异常点，并在 `*_outliers.csv` / `*_outliers_summary.csv` 中记录哪些点被删、每条鱼删了几个
3b. **镜像过滤**：同一帧内若两点 y 相近且 x 关于某轴对称（个体与玻璃/水面镜像），保留与上一帧更一致的点，另一者置为 NaN，并写入 `*_mirror_outliers.csv`
4. **缺失值填充**：插值补全，保证每一帧都有 3 个个体坐标
5. **可视化**：ID 分配后与最终处理后的轨迹图

跨视频的 **唯一 ID_1 / ID_2 / ID_3** 在 **06 3D 轨迹** 中通过 Step 5 统一。

详见 `scripts/tracking/README.md` 与 `ALGORITHM_DESCRIPTION.md`。

In [ ]:
# 路径与配置（下面两行直接改即可；打印时盘符统一大写）
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

SLEAP_OUTPUT_DIR = Path(r"D:\sleap_output")
PROCESSED_DIR = SLEAP_OUTPUT_DIR / "processed"

def _norm(p):
    s = str(Path(p).resolve())
    if len(s) >= 2 and s[1] == ":":
        return s[0].upper() + s[1:]
    return s
print("PROJECT_ROOT:", _norm(PROJECT_ROOT))
print("SLEAP_OUTPUT_DIR:", _norm(SLEAP_OUTPUT_DIR))
print("PROCESSED_DIR:", _norm(PROCESSED_DIR))

## ID 分配 → 异常值处理 →（可选）缺失值填充

可传入**多组** PLAYER1 的 CSV 路径（列表 `csv_paths`），会按组依次处理；每组会自动查找同 set 的 PLAYER2/3，并生成 `file_sets.json`（若不存在）。

**注意**：当 `skip_2d_interpolation=True`（默认）时，**不**做 2D 缺失值填充，异常点剔除后缺帧保留为 NaN，留到 05/06 的 3D 阶段再插值。


In [ ]:
# 支持多组：传入多个 PLAYER1 的 CSV 路径（列表），会按组依次处理
from scripts.tracking.process_tracking import process_from_csv_path

# 每组填一个 PLAYER1 的 CSV；可填多组，一次跑完
csv_paths = [
    r"D:\sleap_output\top_mouth_180\20251225_K_1_2_clips\20251225_K_1_2_clips_20251225_K_1_2_FCMC_PLAYER1_m1tf_Courtship_290.729-296.331.csv",
    r"D:\sleap_output\top_mouth_180\20251225_K_1_2_clips\20251225_K_1_2_clips_20251225_K_1_2_FCMC_PLAYER1_m1tf_Wrap_387.663-395.277.csv",
    r"D:\sleap_output\top_mouth_180\20251225_K_1_2_clips\20251225_K_1_2_clips_20251225_K_1_2_FCMC_PLAYER1_m2tm1_Dash_1209.175-1213.349.csv",
]
# 若只处理一组，也可写成 csv_paths = [r"一条路径"]
if isinstance(csv_paths, str):
    csv_paths = [csv_paths]

sleap_output_dir = str(SLEAP_OUTPUT_DIR)
output_dir = str(PROCESSED_DIR)
use_frame_gap_scaling = True
max_jump_per_frame = 100
max_jump_distance = 50
flow_weight = 10.0
max_iterations = 50   # 2D 异常值循环：检测→剔除→插值 的最大轮数，仍有余点会打警告
apply_mirror_filter = True  # 是否执行步骤3b镜像过滤（个体与镜像成对时删镜像点）

for i, csv_path in enumerate(csv_paths, 1):
    if not csv_path or not Path(csv_path).exists():
        print(f"[{i}/{len(csv_paths)}] 跳过（不存在）: {csv_path}")
        continue
    print(f"\n[{i}/{len(csv_paths)}] 将处理: {csv_path}")
    result = process_from_csv_path(
        csv_path,
        sleap_output_dir,
        output_dir=output_dir,
        max_distance=50,
        max_jump_distance=max_jump_distance,
        fps=166,
        interp_method="linear",
        interp_limit=5,
        smooth=True,
        visualize=True,
        use_frame_gap_scaling=use_frame_gap_scaling,
        max_jump_per_frame=max_jump_per_frame,
        flow_weight=flow_weight,
        max_iterations=max_iterations,
        apply_mirror_filter=False,
        skip_2d_interpolation=True,  # 目前关闭 2D 插值：outlier 直接删掉，缺帧留 NaN（后续如需可在 3D 再插）
    )
print("\n全部处理完成，结果在:", output_dir)

## 2D 轨迹叠加原视频并导出 MP4

将**已分配 ID 并处理后的 2D 轨迹**（只使用上一步生成的 `*_player[123]_processed.csv` 文件，不再使用 `*_id_assigned.csv` 或原始 SLEAP CSV）叠到**原 clip 视频**上，生成三视角并排的 MP4：上方为原视频 + 当前帧个体位置与 ID，下方为最近一段时间内的轨迹尾迹。  
依赖：`opencv-python`。需指定**原视频所在目录**（或分别指定三个视角视频路径），脚本会按文件名匹配对应视角。

In [ ]:
# 2D 轨迹叠加原视频 → MP4（使用 scripts/analysis/render_tracking_video 逻辑）
import re
from pathlib import Path

from scripts.analysis.render_tracking_video import (
    find_view_csvs,
    load_tracking_data,
    normalise_individual_to_int,
    render_three_view_video,
)

# 要导出的 set：与上方处理的 set_key 一致（即某条 csv_path 对应的 set）
SET_KEY_VIDEO = "20251225_K_1_2_clips_20251225_K_1_2_FCMC_m1tf_Courtship_290.729-296.331"
# 已处理 CSV 所在目录（与 PROCESSED_DIR 一致即可）
INPUT_CSV_DIR = PROCESSED_DIR
# 原 clip 视频根目录（会在此目录及子目录下按「时间段 + PLAYER1/2/3」自动查找三路视频）
VIDEO_DIR = Path(r"D:\behavior_clips\20251225_K_1_2_FCMC_clips")
# 若已明确知道三路视频路径，可在此指定，否则留 None 走自动查找
VIDEO_TOP = None
VIDEO_SIDE1 = None
VIDEO_SIDE2 = None

FPS_VIDEO = 166
TRAIL_DURATION_S = 0.5   # 下方轨迹尾迹保留时长（秒）
OUTPUT_MP4 = PROCESSED_DIR / f"{SET_KEY_VIDEO}_2d_tracking_overlay.mp4"


def find_videos_in_dir(video_dir: Path, set_key: str, video_top=None, video_side1=None, video_side2=None):
    """在 video_dir 下按 set_key 中的时间段（如 290.729-296.331）和 PLAYER1/2/3 自动找三路 mp4。"""
    out = {}
    if video_top and Path(video_top).exists():
        out["top"] = Path(video_top)
    if video_side1 and Path(video_side1).exists():
        out["side1"] = Path(video_side1)
    if video_side2 and Path(video_side2).exists():
        out["side2"] = Path(video_side2)
    if len(out) == 3:
        return out
    m = re.search(r"(\d+\.\d+-\d+\.\d+)", set_key)
    time_part = m.group(1) if m else ""
    if not video_dir.exists() or not time_part:
        return out
    all_mp4 = list(video_dir.rglob("*.mp4"))
    def by_view(name_upper):
        if "PLAYER1" in name_upper:
            return "top"
        if "PLAYER2" in name_upper:
            return "side1"
        if "PLAYER3" in name_upper:
            return "side2"
        return None
    for p in all_mp4:
        if time_part not in p.stem:
            continue
        v = by_view(p.stem.upper())
        if v and v not in out:
            out[v] = p.resolve()
    return out


view_csvs = find_view_csvs(Path(INPUT_CSV_DIR), SET_KEY_VIDEO)
video_paths = find_videos_in_dir(VIDEO_DIR, SET_KEY_VIDEO, VIDEO_TOP, VIDEO_SIDE1, VIDEO_SIDE2)
for v in ["top", "side1", "side2"]:
    if v in view_csvs and v not in video_paths:
        raise FileNotFoundError(
            f"未找到视角 {v} 的视频。已在 {VIDEO_DIR} 下按 set_key 时间段与 PLAYER1/2/3 匹配；"
            f"请确认该目录内有含对应时间段且文件名含 PLAYER1/PLAYER2/PLAYER3 的 .mp4。"
        )

view_data = {}
for v, csv_path in view_csvs.items():
    df = load_tracking_data(csv_path)
    if "fish_id" in df.columns:
        ind = df[["frame", "fish_id", "x", "y"]].copy()
        ind = ind.rename(columns={"fish_id": "individual"})
    elif "individual" in df.columns:
        ind = df[["frame", "individual", "x", "y"]].copy()
    else:
        from scripts.analysis.render_tracking_video import extract_centroid_per_frame
        ind = extract_centroid_per_frame(df)
    ind["individual"] = ind["individual"].apply(normalise_individual_to_int)
    ind = ind[ind["individual"] >= 0]
    if ind.empty:
        continue
    view_data[v] = ind

if not view_data:
    raise SystemExit("没有视角得到有效轨迹")
vp = {v: video_paths[v] for v in view_data if v in video_paths}
if len(vp) != len(view_data):
    raise SystemExit(f"缺少视角视频: {set(view_data) - set(vp)}")

OUTPUT_MP4.parent.mkdir(parents=True, exist_ok=True)
render_three_view_video(
    view_data, vp, OUTPUT_MP4, FPS_VIDEO,
    trail_duration_s=TRAIL_DURATION_S,
    show_trail_row=False,   # 不显示下方纯轨迹行
    simple_dots=True,       # 仅纯色圆点，不画 Ind_0 等文字
)

## 线性插值补全各 CSV

对上述步骤生成的 `*_player*_processed.csv` 中的缺失值（NaN）做**线性插值**，使每条轨迹在每帧都有坐标，便于后续 3D 重建与可视化。  
（当 `skip_2d_interpolation=True` 时，上一步未做 2D 插值，此处统一补全。）

In [ ]:
# 扫描 processed 目录下所有 *_player*_processed.csv，用线性插值补全缺失值并写回
import pandas as pd
from pathlib import Path

from scripts.tracking.missing_value_filling import fill_missing_values_trajectory

# 与上方配置一致
INTERP_METHOD = "linear"
INTERP_LIMIT = 5   # 最大连续缺失帧数（超过则不插值）；None 表示不限制
SMOOTH = True

processed_dir = Path(PROCESSED_DIR)
# 按 player1 的 processed 文件找齐各 set
player1_processed = list(processed_dir.glob("*_player1_processed.csv"))
if not player1_processed:
    print("未找到任何 *_player1_processed.csv，请先运行上方「ID 分配 → 异常值处理」单元。")
else:
    for p1_path in sorted(player1_processed):
        set_key = p1_path.stem.replace("_player1_processed", "")
        paths = {
            "player1": processed_dir / f"{set_key}_player1_processed.csv",
            "player2": processed_dir / f"{set_key}_player2_processed.csv",
            "player3": processed_dir / f"{set_key}_player3_processed.csv",
        }
        for player, path in paths.items():
            if not path.exists():
                print(f"跳过 set {set_key}：缺少 {path.name}")
                continue
            df = pd.read_csv(path)
            if "fish_id" not in df.columns:
                print(f"跳过 {path.name}：无 fish_id 列")
                continue
            fish_ids = [f for f in df["fish_id"].unique() if f != "unassigned"]
            for fid in fish_ids:
                df = fill_missing_values_trajectory(
                    df, fid,
                    method=INTERP_METHOD,
                    interp_limit=INTERP_LIMIT,
                    smooth=SMOOTH,
                )
            df.to_csv(path, index=False)
            print(f"已插值并写回: {path}")
    print("\n线性插值补全完成。")